# 010_provision_core_artifacts
Creates the lakehouse folders for each layer and the Spark databases. Business-schema
databases (`bronze_wahat` and so on) are also created on demand by the loaders.
Safe to re-run.

In [ ]:
# Bootstrap: find the pipeline root and load the shared notebooks.
# run_notebook() plays the part of Databricks' %run: it runs another notebook's code cells in
# this notebook's namespace. It is used instead of Jupyter's %run because that breaks on
# Windows paths with spaces.
import json
import pathlib

PIPELINE_ROOT = next(
    folder
    for folder in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (folder / "00_shared").is_dir()
)


def run_notebook(relative_path):
    notebook = json.loads((PIPELINE_ROOT / relative_path).read_text(encoding="utf-8"))
    for cell in notebook["cells"]:
        if cell["cell_type"] == "code":
            exec("".join(cell["source"]), globals())


if "SHARED_LOADED" not in globals():
    run_notebook("00_shared/000_nb_configurations.ipynb")
    run_notebook("00_shared/001_nb_utils.ipynb")
    SHARED_LOADED = True

## 1. Create the layer folders and databases

In [ ]:
BUSINESS_SCHEMAS = ["wahat"]

for layer in LAYERS:
    for schema in BUSINESS_SCHEMAS:
        (LAKEHOUSE / layer / schema).mkdir(parents=True, exist_ok=True)
        spark.sql(f"CREATE DATABASE IF NOT EXISTS {layer_database(layer, schema)}")
        print(f"{layer_database(layer, schema):<16} ready  ({LAKEHOUSE / layer / schema})")

CTRL_FOLDER.mkdir(parents=True, exist_ok=True)
PUBLISH_FOLDER.mkdir(parents=True, exist_ok=True)
spark.sql(f"CREATE DATABASE IF NOT EXISTS {CTRL_DATABASE}")
print(f"{CTRL_DATABASE:<16} ready  ({CTRL_FOLDER})")